In [2]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from ingest import load_faq_data
documents = load_faq_data()

In [3]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

84

In [4]:
documents = documents_llm


In [5]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [6]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [7]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [9]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [10]:
import json

user_prompt = json.dumps(doc)

In [11]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [13]:
response = openai_client.responses.parse(
    model="openai/gpt-oss-120b",
    input=messages,
    text_format=Questions
)

In [14]:
result = response.output_parsed

print(result)

questions=['I just found out about the LLM Zoomcamp, is it still possible for me to join the class?', 'If I enroll late, will I still be able to get the course certificate?', 'Do I have to submit a project to receive the certificate, even if I join after the start date?', 'Are there any deadlines for submitting the final project if I sign up now?', 'Can I access all the course materials and labs after the official start date?']


In [15]:
print(result.questions)


['I just found out about the LLM Zoomcamp, is it still possible for me to join the class?', 'If I enroll late, will I still be able to get the course certificate?', 'Do I have to submit a project to receive the certificate, even if I join after the start date?', 'Are there any deadlines for submitting the final project if I sign up now?', 'Can I access all the course materials and labs after the official start date?']


In [16]:
from evaluation_utils import llm_structured

In [ ]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions,
    model="openai/gpt-oss-120b"
)

print(result.questions)

['Can I enroll in the LLM Zoomcamp now that it already started?', 'Is there a deadline to submit my final project if I join late?', 'Will I still be able to get a certificate if I start the course late?', 'What are the current submission dates for the course projects?', 'Do I need to finish the whole curriculum to earn the certificate?']


In [19]:
usage.input_tokens, usage.output_tokens

(336, 348)

In [20]:
from evaluation_utils import calc_price


In [21]:
cost = calc_price(usage)

cost

{'input_cost': 0.000252,
 'output_cost': 0.0015660000000000001,
 'total_cost': 0.0018180000000000002}

In [22]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I enroll in the LLM Zoomcamp now that it already started?',
  'document': '74eb249bbf'},
 {'question': 'Is there a deadline to submit my final project if I join late?',
  'document': '74eb249bbf'},
 {'question': 'Will I still be able to get a certificate if I start the course late?',
  'document': '74eb249bbf'},
 {'question': 'What are the current submission dates for the course projects?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to finish the whole curriculum to earn the certificate?',
  'document': '74eb249bbf'}]

In [23]:
from evaluation_utils import llm_structured_retry


In [26]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model="openai/gpt-oss-120b"
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [27]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [28]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

In [ ]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

In [29]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.010133250000000002

In [30]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [32]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)
